## Graph Neural Network

GNNs are deep learning methods that work on graph-structured data. This family of methods is also known as **geometric deep learning** and is gaining increasing interest in a variety of applications, including social network analysis and computer graphics. Different variants of the GNN have been proposed, with the aim of improving its representation learning capability:

- Some of them are specifically designed to process **specific types of graphs** (directed, undirected, weighted, unweighted, static, dynamic, and so on).

- Also, several modifications have been proposed for **the propagation step** (convolution, gate mechanisms, attention mechanisms, and skip connections, among others), with the aim of improving representation at different levels. Also, different training methods have been proposed to improve learning stability and generalization, including supervised, semi-supervised, and self-supervised approaches.

- In addition, efforts have been made to address **scalability challenges** when applying GNNs to large-scale graphs, such as through sampling strategies, layer-wise neighborhood aggregation, and distributed training.

- Another line of research focuses on **model expressiveness and depth**, investigating how deeper GNN architectures can capture more complex patterns without suffering from over-smoothing or vanishing gradients.

- Furthermore, **domain-specific adaptations** of GNNs have emerged for tasks in bioinformatics, recommender systems, knowledge graphs, and traffic forecasting, each incorporating inductive biases relevant to their data structure.

Overall, the evolution of GNN architectures reflects a growing emphasis on balancing expressive power, computational efficiency, and adaptability to diverse application domains. As the field matures, further research continues to explore novel mechanisms for aggregation, positional encoding, and integration with other learning paradigms such as reinforcement learning and generative modeling.


In the context of unsupervised representation learning on graphs, a widely adopted approach involves employing an **encoder–decoder architecture**. The encoder—typically implemented using a variant of a Graph Neural Network (GNN)—is responsible for generating low-dimensional node embeddings that capture the structural and semantic properties of the graph. The decoder, on the other hand, attempts to reconstruct the original graph structure, often represented by the adjacency matrix, from these embeddings.

The training objective is commonly designed to minimize the discrepancy between the original adjacency matrix and the one reconstructed by the decoder. This is often expressed as a similarity or distance-based loss function that encourages the embeddings to preserve local and global graph topology. A typical formulation of this loss function is:
$$
\mathcal{L} = \sum_{i,j} \left( A_{ij} - \hat{A}_{ij} \right)^2
$$
Formally, this framework can be defined as follows:

$$
\mathbf{Z} = \text{GNN}(\mathbf{X}, \mathbf{A}), \quad \hat{\mathbf{A}} = \mathbf{Z} \mathbf{Z}^\top
$$

Here, $ \mathbf{A} \in \mathbb{R}^{n \times n} $ denotes the adjacency matrix of the graph, and  
$ \mathbf{X} \in \mathbb{R}^{n \times d} $ represents the node attribute matrix, where  
$ n$ is the number of nodes and $d $ is the dimensionality of node features.  
The encoder, implemented as a GNN, maps the input features and structure to a latent representation  
$ \mathbf{Z} \in \mathbb{R}^{n \times h} $, where $ h $ is the embedding dimension.  
The reconstructed adjacency matrix $ \hat{\mathbf{A}}$ is obtained by computing the similarity between node embeddings, often using the inner product.

This unsupervised framework provides a powerful means of learning meaningful node representations that can be effectively utilized for downstream tasks such as node classification, clustering, and link prediction, without requiring labeled data.


Another common variant of this approach, especially used in graph-level representation learning (e.g., for graph classification), is to train the model using a distance-based loss. In this setting, the model embeds a pair of graphs simultaneously, and the learned representations are optimized to reflect a given similarity or distance metric between the graphs. This strategy encourages the embeddings of similar graphs to be close in the latent space, and dissimilar graphs to be far apart.

A similar technique can be applied to node-level tasks as well, where instead of whole-graph distances, a node similarity function is used as the learning signal. In such cases, the goal is to train the model so that the distance (or similarity) between embeddings of node pairs reflects their structural or semantic similarity in the original graph. Contrastive losses and ranking-based objectives are commonly used to support this training paradigm.

**Graph Convolutional Neural Network (GCN)**-based encoders are among the most widely used variants of GNNs for unsupervised representation learning. GCNs are inspired by many of the core principles of Convolutional Neural Networks (CNNs). In GCNs, filter parameters are typically shared across all nodes in the graph, enabling parameter efficiency and weight tying. Multiple graph convolutional layers are stacked to build deep architectures that can capture increasingly abstract representations.

There are essentially two main types of convolutional operations applied to graph-structured data:

- **Spectral approaches**: These methods **define convolution in the spectral domain by leveraging the eigen decomposition of the graph Laplacian**. The idea is to interpret convolution as a filtering operation applied in the frequency domain, where graph signals are decomposed into a combination of orthogonal components. Although theoretically elegant, spectral methods often suffer from high computational cost and limited transferability across different graph structures.

- **Spatial approaches**: In contrast, spatial methods **define convolution directly in the node domain. They operate by aggregating feature information from a node's local neighborhood**. This aggregation is typically performed through learnable functions such as weighted sums, mean pooling, or attention mechanisms. Spatial methods are more flexible and scalable, and have become the dominant choice in modern GNN architectures.

Overall, both spectral and spatial approaches aim to exploit the locality and structure of graphs to learn meaningful node and graph-level representations. However, spatial methods tend to be more adaptable and practical in real-world applications.


# Spectral Graph Convolution in Unsupervised Learning

## 1. Graph Preliminaries

Let $G = (V, E)$ be an undirected graph with:

- $ n = |V| $ nodes  
- Adjacency matrix $ \mathbf{A} \in \mathbb{R}^{n \times n} $  
- Degree matrix $ \mathbf{D} \in \mathbb{R}^{n \times n} $, where $ D_{ii} = \sum_j A_{ij} $  
- Node feature matrix $ \mathbf{X} \in \mathbb{R}^{n \times d} $

The **unnormalized Laplacian** is:

$$
\mathbf{L} = \mathbf{D} - \mathbf{A}
$$
So $ \mathbf{L} $ is the graph Laplacian matrix.
- Perform eigen decomposition of $ \mathbf{L} $:  
 $$
  \mathbf{L} = \mathbf{U} \boldsymbol{\Lambda} \mathbf{U}^\top
 $$
  where:
  - $ \mathbf{U} $ contains eigenvectors as columns.
  - $ \boldsymbol{\Lambda} $ is a diagonal matrix with eigenvalues.

- The Graph Fourier Transform projects the signal $ \mathbf{x} $ into the graph spectral domain (eigenvectors base):
  $$
  \hat{\mathbf{x}} = \mathbf{U}^\top \mathbf{x}
  $$
  
  This is analogous to the classical Fourier transform which projects signals onto sine and cosine bases.


 To recover the original signal from the spectral domain:
  $$
  \mathbf{x} = \mathbf{U} \hat{\mathbf{x}}
  $$

- Multiplying by $ \mathbf{U} $ reconstructs the signal from its spectral representation.

---

### Spectral Convolution Definition

- The convolution of a signal $ \mathbf{x} $ with a filter $g_\theta $ is defined as:
  $$
  g_\theta * \mathbf{x} = \mathbf{U} \, g_\theta(\boldsymbol{\Lambda}) \, \mathbf{U}^\top \mathbf{x}
  $$

- Interpretation:
  - Transform $ \mathbf{x} $ to spectral domain: $ \mathbf{U}^\top \mathbf{x} $.
  - Apply the filter $ g_\theta(\boldsymbol{\Lambda}) $, which is a function of eigenvalues (a diagonal matrix).
  - Transform back to the node domain with $ \mathbf{U} $.

---

### Chebyshev Polynomial Approximation

The Chebyshev polynomials $ T_k(x) $ are defined recursively as follows:

- $ T_0(x) = 1 $
- $ T_1(x) = x $
- For $k \geq 2 $:  $T_k(x) = 2x \cdot T_{k-1}(x) - T_{k-2}(x)$

- Direct computation of $ g_\theta(\boldsymbol{\Lambda}) $ can be expensive.
- Instead, approximate $ g_\theta(\mathbf{L}) $ with a truncated expansion of Chebyshev polynomials:
  $$
  g_\theta(\mathbf{L}) \mathbf{x} \approx \sum_{k=0}^K \theta_k T_k(\tilde{\mathbf{L}}) \mathbf{x}
  $$

- Where:
  - $ \tilde{\mathbf{L}} = \frac{2}{\lambda_{\max}} \mathbf{L} - \mathbf{I} $ rescales the Laplacian eigenvalues to lie within $[-1,1]$, the domain of Chebyshev polynomials.
  - $ T_k $ is the Chebyshev polynomial of degree $ k$, defined recursively and with good numerical properties.
  - $ \theta_k$ are learnable parameters (weights of the filter).
  - $ K $ is the polynomial degree that controls the filter’s locality (how far the convolution spreads in the graph).

While this approximation is efficient, Kipf & Welling proposed a **simplified version** for practical and scalable deep learning on graphs.

---

### First-Order Approximation of Chebyshev Expansion

To reduce computational complexity further, they used:

- $ K = 1 $ (i.e., only the first two Chebyshev polynomials: $ T_0(x) = 1 $ and $ T_1(x) = x $).
- This gives the simplified form:
  
  $$
  g_\theta(\mathbf{L}) \mathbf{x} \approx \theta_0 \mathbf{x} + \theta_1 \tilde{\mathbf{L}} \mathbf{x}
  $$

  where:
  $$
  \tilde{\mathbf{L}} = \frac{2}{\lambda_{\max}} \mathbf{L} - \mathbf{I}
  $$

- They set $ \lambda_{\max} \approx 2 $ and combine the parameters $ \theta_0, \theta_1 $ into one shared parameter $ \theta $, leading to:

 $$
  g_\theta * \mathbf{x} \approx \theta (\mathbf{I} + \mathbf{D}^{-1/2} \mathbf{A} \mathbf{D}^{-1/2}) \mathbf{x}
 $$



### GCN Layer Formulation

To ensure numerical stability and include self-loops, the following **normalized adjacency matrix with self-loops** is used:

1. ##### Add self-loops:
  $$
   \tilde{\mathbf{A}} = \mathbf{A} + \mathbf{I}
   $$
---
   This means each node is now connected to itself in addition to its neighbors. This has several advantages:



##### 1-1. Preserves Node's Own Features During Aggregation

In a GCN layer:

$$
\mathbf{H}^{(l+1)} = \sigma \left( \hat{\mathbf{A}} \mathbf{H}^{(l)} \mathbf{W}^{(l)} \right)
$$

- Without self-loops: A node $ i $'s representation is updated only using its neighbors’ features.
- With self-loops: A node also contributes **its own features** to its update.
- This ensures:

$$
h_i^{(l+1)} \leftarrow h_i^{(l)} + \text{neighbors' features}
$$

##### 1-2. Improves Expressiveness and Handles Isolated Nodes

- Nodes with no neighbors (isolated nodes) would receive **zero information** without self-loops.
- Adding self-loops ensures **every node has at least one connection** — to itself.
- Makes the model robust to sparse or irregular graph structures.

##### 1-3. Stabilizes Training and Prevents Oversmoothing

- Deep GCNs can suffer from **oversmoothing**, where node embeddings become indistinguishable.
- Self-loops **retain the identity** of each node across layers.
- Helps balance **self-information vs. neighbor information**, improving gradient flow.


##### 1-4. Theoretical Justification in Spectral Domain

- Adding self-loops modifies the Laplacian spectrum.
- Leads to better **numerical stability** in spectral filtering.
- Prevents issues like very large/small eigenvalues which can distort message passing.

---

##### 2. Compute the corresponding degree matrix:
   $$
   \tilde{\mathbf{D}}_{ii} = \sum_j \tilde{\mathbf{A}}_{ij}
   $$

##### 3. Define the **symmetric normalized adjacency matrix**:
   $$
   \hat{\mathbf{A}} = \tilde{\mathbf{D}}^{-1/2} \tilde{\mathbf{A}} \tilde{\mathbf{D}}^{-1/2}
   $$

##### 4. Now, a single **GCN layer** is defined as:
   $$
   \mathbf{H}^{(l+1)} = \sigma \left( \hat{\mathbf{A}} \mathbf{H}^{(l)} \mathbf{W}^{(l)} \right)
   $$

   Where:
   - $ \mathbf{H}^{(l)} $ is the input feature matrix at layer $ l $ (with $ \mathbf{H}^{(0)} = \mathbf{X} $, the raw features).
   - $ \mathbf{W}^{(l)} $ is the learnable weight matrix.
   - $ \sigma$ is an activation function (like ReLU).

---

####  Stacking GCN Layers

You can stack multiple GCN layers to obtain deeper representations:

$$
\mathbf{Z} = \text{softmax} \left( \hat{\mathbf{A}} \, \sigma \left( \hat{\mathbf{A}} \mathbf{X} \mathbf{W}^{(0)} \right) \mathbf{W}^{(1)} \right)
$$

- $ \mathbf{Z} \in \mathbb{R}^{n \times C} $ is the matrix of class probabilities (in semi-supervised classification).
- $ C $ is the number of classes.
- The first layer extracts representations, and the second layer projects them into the output space.

---

### Summary: Transition from Spectral to GCN

| Step | Description |
|------|-------------|
| 1 | Start with spectral convolution using Laplacian eigen decomposition |
| 2 | Use Chebyshev approximation to make it computationally feasible |
| 3 | Apply a first-order approximation (K = 1) for further simplification |
| 4 | Introduce self-loops and normalized adjacency for stability |
| 5 | Final GCN layer: $ \mathbf{H}^{(l+1)} = \sigma(\hat{\mathbf{A}} \mathbf{H}^{(l)} \mathbf{W}^{(l)}) $ |







---

## 3. Graph Autoencoder (Unsupervised)

We use:

- Encoder: Spectral GCN  
- Decoder: Reconstruct  
$
\hat{\mathbf{A}} = \sigma(\mathbf{Z} \mathbf{Z}^\top)
$ 
- Loss: Reconstruction loss (e.g., binary cross-entropy)


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def normalize_adj(A):
    """Compute D^(-1/2) A D^(-1/2)"""
    A = A + torch.eye(A.size(0))
    D = torch.diag(torch.sum(A, dim=1))
    D_inv_sqrt = torch.linalg.inv(torch.sqrt(D))
    return D_inv_sqrt @ A @ D_inv_sqrt

class SpectralGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Parameter(torch.randn(in_dim, out_dim) * 0.01)

    def forward(self, X, A_norm):
        return A_norm @ X @ self.W

class GraphAutoencoder(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.encoder = SpectralGCNLayer(in_dim, hidden_dim)

    def forward(self, X, A_norm):
        Z = self.encoder(X, A_norm)
        A_hat = torch.sigmoid(Z @ Z.T)
        return A_hat, Z

# Example usage
n_nodes = 100
in_dim = 16
hidden_dim = 8

X = torch.randn(n_nodes, in_dim)
A = torch.randint(0, 2, (n_nodes, n_nodes)).float()
A = (A + A.T) / 2
A[A > 0] = 1
A.fill_diagonal_(0)

A_norm = normalize_adj(A)

model = GraphAutoencoder(in_dim, hidden_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

for epoch in range(100):
    model.train()
    A_hat, Z = model(X, A_norm)
    loss = loss_fn(A_hat, A)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 0.6931
Epoch 10, Loss: 0.6660
Epoch 20, Loss: 0.5982
Epoch 30, Loss: 0.5633
Epoch 40, Loss: 0.5675
Epoch 50, Loss: 0.5617
Epoch 60, Loss: 0.5619
Epoch 70, Loss: 0.5609
Epoch 80, Loss: 0.5607
Epoch 90, Loss: 0.5605


# Spatial Graph Convolution in Unsupervised Learning

---

## Overview

Unsupervised learning on graphs aims to learn **node embeddings** without labeled data. These embeddings capture the structure and features of the graph and can be used for tasks like **clustering**, **visualization**, and **link prediction**.

This tutorial focuses on **spatial-based GNNs**, especially **GraphSAGE**, and how they are used in **unsupervised node representation learning**.

---

## Problem Setting

Let:
- $ G = (V, E) $ be an undirected graph
- $ \mathbf{X} \in \mathbb{R}^{|V| \times d} $ be the node feature matrix
- $ \mathcal{N}(v)$ denote the neighbors of node $ v $

Goal:
> Learn a function $ f: V \to \mathbb{R}^k $ that maps nodes to low-dimensional embeddings $ \mathbf{z}_v \in \mathbb{R}^k $ **without supervision**.

---

## Spatial Graph Neural Networks

In **spatial GNNs**, convolution is defined as **neighborhood aggregation**:

### General Spatial Aggregation Rule

For node $ v $ at layer $ l+1 $:

$$
\mathbf{h}_v^{(l+1)} = \sigma \left( \mathbf{W}^{(l)} \cdot \text{AGGREGATE}^{(l)} \left( \left\{ \mathbf{h}_v^{(l)} \right\} \cup \left\{ \mathbf{h}_u^{(l)}, \forall u \in \mathcal{N}(v) \right\} \right) \right)
$$

- $ \mathbf{h}_v^{(0)} = \mathbf{x}_v $
- $ \mathbf{W}^{(l)} $ is a learnable weight matrix
- $ \sigma $ is an activation function (e.g., ReLU)
- `AGGREGATE` can be `mean`, `max`, `LSTM`, etc.

---

## GraphSAGE: Neighborhood Sampling & Aggregation

GraphSAGE (Hamilton et al., 2017) uses a **sampling-based** neighborhood aggregation method.

### GraphSAGE Mean Aggregator

$$
\mathbf{h}_v^{(l+1)} = \sigma \left( \mathbf{W}^{(l)} \cdot \left[ \mathbf{h}_v^{(l)} \, \Vert \, \text{MEAN}\left( \left\{ \mathbf{h}_u^{(l)}, \forall u \in \mathcal{N}(v) \right\} \right) \right] \right)
$$

- $ \Vert $ denotes concatenation
- MEAN aggregates over neighbors' features
- Multiple layers stack to capture higher-order structure

---

## Unsupervised Training Objective: Contrastive Learning

We train the model to **maximize similarity between embeddings of nearby nodes**, and **minimize similarity with random nodes**.

### Negative Sampling Loss (SkipGram-style)

Given an anchor node $ v $, a positive context node $ u \in \mathcal{N}_k(v) $, and negative samples $ u' \sim P_n $, define:

$$
\mathcal{L}_v = -\log \left( \sigma(\mathbf{z}_v^\top \mathbf{z}_u) \right) - \sum_{u'} \log \left( \sigma(-\mathbf{z}_v^\top \mathbf{z}_{u'}) \right)
$$

- $ \sigma(x) = \frac{1}{1 + \exp(-x)} $
- $ \mathcal{N}_k(v) $ is a sampled k-hop neighborhood
- $ P_n $: noise distribution (e.g., uniform over nodes)

This is similar to **word2vec** training.

---




In [3]:
## 🔧 Implementation with PyTorch Geometric

import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import negative_sampling

# Load Cora dataset
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

# Define GraphSAGE encoder
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

# Contrastive loss: similar to DeepWalk or node2vec
def unsupervised_loss(z, edge_index):
    pos_edge_index = edge_index
    neg_edge_index = negative_sampling(edge_index, num_nodes=z.size(0))

    pos_score = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(dim=1)
    neg_score = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=1)

    loss = -F.logsigmoid(pos_score).mean() - F.logsigmoid(-neg_score).mean()
    return loss

# Training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GraphSAGE(dataset.num_features, 128, 64).to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

model.train()
for epoch in range(1, 201):
    optimizer.zero_grad()
    z = model(data.x, data.edge_index)
    loss = unsupervised_loss(z, data.edge_index)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

Epoch 20, Loss: 1.1196
Epoch 40, Loss: 1.0137
Epoch 60, Loss: 0.9222
Epoch 80, Loss: 0.8974
Epoch 100, Loss: 0.8680
Epoch 120, Loss: 0.8661
Epoch 140, Loss: 0.8601
Epoch 160, Loss: 0.8657
Epoch 180, Loss: 0.8487
Epoch 200, Loss: 0.8620


## Comparison with Other GNN Types

| GNN Type        | Aggregation Style     | Spectral/Spatial | Good for Unsupervised? |
|-----------------|-----------------------|------------------|-------------------------|
| GCN             | Laplacian-based       | Spectral         | ✅ Yes                  |
| GraphSAGE       | Sampling + Mean/Max   | Spatial          | ✅ Yes (scalable)       |
| GAT             | Attention-based       | Spatial          | ✅ Yes (dense graphs)   |
| GIN             | Sum + MLP             | Spatial          | ❌ Needs supervision    |


Unlike some other GNN frameworks, **GraphSAGE does not define a loss function directly over the adjacency matrix** $ \mathbf{A}$ or its normalized/self-looped variant $ \tilde{\mathbf{A}} $. Here's why:

---

#### 🔸 1. GraphSAGE is Not a Reconstruction-Based Model

GraphSAGE is designed to **learn node embeddings** by aggregating information from sampled neighbors. It is not an autoencoder and does **not aim to reconstruct** the adjacency matrix.

In contrast, **Graph Autoencoders (GAE)** and **Variational Graph Autoencoders (VGAE)** minimize reconstruction loss over $ \mathbf{A} $:

$$
\mathcal{L}_\text{GAE} = \| \sigma(\mathbf{Z} \mathbf{Z}^\top) - \mathbf{A} \|
$$



#### 🔸 2. $ \mathbf{A} $ is Used Operationally, Not as a Target

GraphSAGE uses the graph structure for **sampling neighbors** $ \mathcal{N}(v)$, not for matrix multiplication like spectral methods (e.g., GCN). The model **does not treat $\mathbf{A} $ as a label or prediction target**.


#### 🔸 3. Learning is Based on Contrastive Objectives

GraphSAGE uses an **unsupervised contrastive loss** inspired by word2vec:

$$
\mathcal{L}_v = -\log \sigma(\mathbf{z}_v^\top \mathbf{z}_u) - \sum_{u' \in \text{neg}} \log \sigma(-\mathbf{z}_v^\top \mathbf{z}_{u'})
$$

- $ u $ is a **positive neighbor** (close in the graph)
- $ u' $ is a **negative sample** (random node)
- Encourages embeddings of neighboring nodes to be similar

---

### 📌 Summary Table

| Concept                         | GraphSAGE                              | GAE / VGAE                         |
|----------------------------------|----------------------------------------|------------------------------------|
| Learning target                 | Embedding similarity (contrastive)     | Adjacency matrix reconstruction    |
| Uses adjacency in loss?         | ❌ No                                   | ✅ Yes                              |
| How adjacency is used           | Sampling neighbors                     | Reconstruction & message passing   |
| Loss type                       | SkipGram-style contrastive loss        | Reconstruction loss (e.g., MSE)    |

---
